In [32]:
# ===================== LES DEPENDANCES =================

import os
import torch
from typing import List, Annotated
from langgraph.graph import StateGraph, START, END
import transformers
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage,SystemMessage, ToolMessage,BaseMessage
from langchain_huggingface import HuggingFaceEndpoint
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from tavily import TavilyClient


In [33]:
# ================= CONFIG ================

load_dotenv()



True

In [53]:
load_dotenv()

llm_ggl = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    api_key=os.getenv("GEMINI_API_KEY"),
    streaming=True
    )

In [54]:
load_dotenv()

llm_ibm= init_chat_model(
    model="ibm-granite/granite-3.1-1b-a400m-instruct",
    model_provider="huggingface",
    api_key=os.getenv("HF_TOKEN"),
    temperature=0,
    max_tokens=500,
    streaming=True
)

Loading weights: 100%|██████████| 218/218 [00:00<00:00, 930.93it/s]


In [55]:
from langchain_core.tools import tool
from typing import Optional

# ====================== DEFINIR LES TOOLS ==========================

client = TavilyClient()

@tool
def research_web(query: str) -> str:
    """
    Recherche des informations fiables sur le web.

    Args:
        query: La requête de recherche de l'utilisateur.

    Returns:
        str: Résultats de recherche sous forme de texte.
    """
    response = client.search(
        query=query,
        search_depth="advanced",
        max_results=5
    )

    results = response.get("results", [])

    return "\n\n".join(
        f"- {r['title']}\n  {r['url']}\n  {r['content']}"
        for r in results
    )



@tool
def recit_writer(
    subject: str,
    style: str = "Narratif",
    language: str = "Français",
    length: str = "moyen"
) -> str:
    """
    Génère une histoire ou un récit de qualité.

    Args:
        subject: Sujet ou thème principal de l'histoire.
        style: Style d'écriture (Narratif, Poétique, Dramatique, Humoristique...).
        language: Langue de sortie.
        length: Longueur souhaitée ("court", "moyen", "long").

    Returns:
        L'histoire générée.
    """


    prompt = f""" Tu es un écrivain et narrateur professionnel reconnu pour ton style captivant et ton empathie.

        CONTEXTE :
        Sujet : {subject}
        Style : {style}
        Langue : {language}
        Longueur : {length}

        INSTRUCTIONS :
        1. Adopte un ton {style}.
        2. Commence par une accroche sensorielle (une scène, un son, une pensée) pour plonger immédiatement le lecteur dans l'histoire.
        3. Structure le récit avec un début (exposition), un milieu (tension/évolution) et une fin (résolution ou ouverture).
        4. Utilise un langage fluide et naturel dans la langue {language}.
        5. Si le sujet est délicat (comme le divorce), traite-le avec bienveillance et profondeur psychologique.
        6. Ne pas inclure de texte d'introduction type "Voici votre histoire", commence directement le récit.
        """

    response = llm_ibm.invoke(prompt)

    return response.content

@tool
def image_generate(prompt: str, style: Optional[str] = None) -> str:
    """
    Génère une image à l'aide de Gemini.

    Args:
        prompt: Description détaillée de l'image à générer.
        style: Style artistique optionnel (cinématographique, illustration, réaliste...).

    Returns:
        Chemin ou URL de l'image générée.
    """
    if style:
        full_prompt = f"{prompt}, style {style}, haute qualité, détaillé"
    else:
        full_prompt = prompt

    # return generate_image_gemini(full_prompt)
    return f"https://gemini-generated-image/{hash(full_prompt)}.png"


@tool
def publisher_linkedin(
    image_url: str,
    caption: str,
    hashtags: Optional[str] = None
) -> str:
    """
    Publie une image sur LinkedIn via Zapier ou API.

    Args:
        image_url: URL publique de l'image.
        caption: Légende / texte du post LinkedIn.
        hashtags: Hashtags optionnels (ex: "#IA #Storytelling").

    Returns:
        Statut de la publication.
    """
    full_caption = caption
    if hashtags:
        full_caption += f"\n\n{hashtags}"

    # return publish_to_linkedin(image_path=image_path, caption=full_caption)

    return f"✅ Publication réussie sur LinkedIn !\nImage : {image_url}\nPost : {full_caption[:100]}..."

In [56]:
# ================ PASSER LES TOOLS AUX LLMs ==============

#tools=[research_web, recit_writer,image_generate,publisher_linkedin ]



llm_ibm_tool=llm_ibm.bind_tools(tools=[research_web, recit_writer])
llm_gemi_tool=llm_ggl.bind_tools(tools=[image_generate,publisher_linkedin])

In [ ]:
# ================= L'ETAT DU GRAPH ==============


from typing import Annotated, TypedDict, Literal

from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class State(TypedDict):

    # Historique complet de la conversation
    messages: Annotated[list[BaseMessage], add_messages]

    # Demande utilisateur
    user_input: str
    text_input: str | None
    audio_file: str | None

    # reseach | story | image | publish
    intent: Literal["research", "story", "image", "publish"]

    # Résultat de la recherche
    research: list[str]

    # Histoire générée
    story: str

    # Prompt envoyé au modèle d'image
    image_prompt: str

    # Image générée
    image_url: str

    # Résultat publication
    publish_result: dict[str,str]

In [ ]:
#============= transcription voix-text==========

from openai import OpenAI

client = OpenAI()

def speech_to_text(audio_path:str)-> str:
    """
    Transcrit un fichier audio en texte à l'aide d'OpenAI.

    Args:
        audio_path: le chemin de l'audio a prendre en compte.

    Returns:
        str: text transcrit a partir du fichier audio

    """
    with open(audio_path, "rb") as audio:
      transcription = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=audio
    )
      return transcription.text

# ===== logique use_case

def input_agent(state: State):

    if state.get("audio_path"):

        text = speech_to_text(
            state["audio_path"]
        )

        return {
            "user_text": text,
            "input_type": "audio"
        }


    elif state.get("user_text"):

        return {
            "input_type": "text"
        }


    else:

        raise ValueError(
            "Aucune entrée"
        )

In [ ]:
# ============== CREER LES SOUS-AGENTS ===============

from langgraph.prebuilt import ToolNode

# definir llm avec tool
# definir un system_prompt
# definir un ToolNode

# Node1
def research_agent(state:State)->State:
  """
  utilise les urls definis pour faire des recherches sur internet.

  """

  response=llm_ibm_tool.invoke()

  return f"réponse générée {user_input}"


# Node2
def story_agent(state:State)->State:
  """
  """
  result=llm_ibm_tool.invoke(user_prompt)
  return f"recit generer selon ta demande:{result}"

# Node3
def image_agent(state:State)->State:
  """
  """
  image=llm_gemi_tool.invoke(user_query)
  return f"voici votre:{image}"

# Node4
def publish_agent(state: State)-> State:
  """
  """
  publication=llm_gemi_tool.invoke(image_generate)
  return f"publication effectuée "


In [ ]:
# ========================== FONCTIONS DE ROUTAGE =============

def route_after_research(state:State):
    if state["intent"] == "story":
        return "story"
    else:
        return "image"
    
    
def route_after_story(state:State):
    if state["publish"]:
        return "publish"
    else:
        return END
   
    
def route_after_image(state: State):
    if state["publish"]:
        return "publish"
    else:
        return END

In [ ]:
# ============== CONSTRUIRE LE GRAPH ===============

graph=StateGraph(State)
graph.add_node("research", research_agent)
graph.add_node("story", story_agent )
graph.add_node("image", image_agent)
graph.add_node("publish",publish_agent )

graph.add_edge(START, "research")

graph.add_conditional_edges(
    "research",
    route_after_research,
    {
        "story": "story",
        "image": "image",
    },
)

graph.add_conditional_edges(
    "story",
    route_after_story,
    {
        "publish": "publish",
        END: END,
    },
)

graph.add_conditional_edges(
    "image",
    route_after_image,
    {
        "publish": "publish",
        END: END,
    },
)

graph.add_edge("publish", END)


import uuid
from langgraph.checkpoint.memory import InMemorySaver

memory=InMemorySaver()
builder=graph.compile(checkpointer=memory)

config={
    "configurable":{
        "thread_id":str(uuid.uuid4())
    }
}


# resp=builder.invoke(
#     input={
#         "messages":[
#             HumanMessage(
#                 "génère-moi une histoire sur le football"
#             )
#         ]
#     }, config=config
# )

# print (resp)
